# 🎬 08. End-to-End Traffic Accident Detection Pipeline & Web Demo (Tuần 8)
## Hệ Thống Nhận Diện Tai Nạn Giao Thông Tự Động Từ Video

Chào mừng bạn đến với **Tuần 8 - Giai đoạn Hoàn thiện & Triển khai Demo**!

Notebook này tích hợp toàn bộ các mô-đun đã hoàn thành từ Tuần 1 đến Tuần 7 thành một **Pipeline hoàn chỉnh (End-to-End Orchestrator)**:
1. 🚗 **YOLO11s Detector**: Phát hiện phương tiện giao thông (xe máy, ô tô, xe buýt, xe tải, người đi bộ).
2. 🔄 **ByteTrack Tracker**: Theo dõi đối tượng liên tục qua các khung hình, duy trì Track ID.
3. 📈 **TrajectoryManager & Feature Extractor**: Trích xuất chuỗi 15 đặc trưng động học không-thời gian.
4. 🧠 **Kinematic Branch (LSTM)**: Nhận diện bất thường quỹ đạo (giảm tốc đột ngột, góc xoay văng).
5. 🎥 **Visual Branch (X3D-S)**: Kích hoạt cascade khi có nghi vấn để xác minh va chạm vật lý.
6. 🔗 **Late Fusion (MLP / Weighted)**: Dung hợp điểm số đa phương thức, dập tắt báo động sai.
7. ⏱️ **Temporal Post-Processor**: Khử rung lắc thời gian, Temporal NMS và Cooldown interval.
8. 🌐 **Interactive Web UI (Gradio)**: Giao diện web trực quan cho phép tải video và chỉnh ngưỡng thời gian thực.

In [2]:
# =====================================================================
# 1. THIẾT LẬP MÔI TRƯỜNG & KIỂM TRA HỆ THỐNG
# =====================================================================
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display, HTML

# Đảm bảo đường dẫn module được nhận diện
if "." not in sys.path:
    sys.path.insert(0, ".")

from src.pipeline import AccidentDetectionPipeline

# Kiểm tra cấu hình hệ thống
config_path = "configs/default.yaml"
with open(config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("=" * 60)
print("🛠️ THÔNG TIN CẤU HÌNH PIPELINE HIỆN TẠI:")
print(f"   - Device              : {cfg['system'].get('device', 'cpu')}")
print(f"   - YOLO Model Weights  : {cfg['detector'].get('weights')}")
print(f"   - LSTM Classifier     : Enabled = {cfg['classifiers']['lstm'].get('enabled')}")
print(f"   - X3D Visual Classifier: Enabled = {cfg['classifiers']['visual_x3d'].get('enabled')}")
print(f"   - Fusion Method       : {cfg['fusion'].get('method')}")
print("=" * 60)

ModuleNotFoundError: No module named 'src'

In [ ]:
# =====================================================================
# 2. KHỞI TẠO PIPELINE HOÀN CHỈNH (END-TO-END PIPELINE)
# =====================================================================

pipeline = AccidentDetectionPipeline(config_path)
print("\n✅ Pipeline nhận diện tai nạn đã được nạp thành công vào bộ nhớ!")
print(f"   - Trạng thái Detector : {type(pipeline.detector).__name__}")
print(f"   - Trạng thái Tracker  : {type(pipeline.trajectory_manager).__name__}")
print(f"   - Trạng thái LSTM     : {'Sẵn sàng' if pipeline.lstm_classifier else 'Chế độ Rule-based dự phòng'}")
print(f"   - Trạng thái X3D      : {'Sẵn sàng' if pipeline.visual_classifier else 'Không kích hoạt'}")
print(f"   - Trạng thái Fusion   : {type(pipeline.fusion).__name__} (Method: {pipeline.fusion.method})")

In [ ]:
# =====================================================================
# 3. CHẠY PHÂN TÍCH TRÊN VIDEO MẪU (INFERENCE RUN)
# =====================================================================
import time
from tqdm.auto import tqdm

input_video = "demo/sample_videos/sample_traffic.mp4"
output_video = "runs/demo_result.mp4"

if not os.path.exists(input_video):
    print(f"⚠️ Không tìm thấy file video mẫu tại: {input_video}")
    print("   Bạn có thể đặt bất kỳ file video giao thông nào (MP4) vào thư mục 'demo/sample_videos/'.")
else:
    print(f"🎬 Bắt đầu phân tích video: {input_video}")
    start_time = time.time()
    
    # Chạy toàn bộ pipeline trên video
    events = pipeline.process_video(input_video, output_path=output_video)
    elapsed = time.time() - start_time
    
    print(f"\n⏱️ Thời gian xử lý: {elapsed:.2f} giây")
    print(f"💾 Video kết quả kèm khung cảnh báo đã lưu tại: {output_video}")
    
    # Hiển thị bảng tổng kết các vụ va chạm / sự cố phát hiện được
    if events:
        df_events = pd.DataFrame(events)
        print(f"\n🚨 PHÁT HIỆN {len(events)} SỰ CỐ GIAO THÔNG:")
        display(df_events[['event_id', 'timestamp_str', 'level', 'score', 'reasons', 'tracks']])
    else:
        print("\n🟢 Video lưu thông bình thường, không phát hiện va chạm nào!")

In [ ]:
# =====================================================================
# 4. TRÍCH XUẤT VÀ HIỂN THỊ CÁC FRAME CẢNH BÁO TIÊU BIỂU
# =====================================================================

if os.path.exists(output_video) and len(events) > 0:
    cap = cv2.VideoCapture(output_video)
    alert_frames = []
    
    for ev in events[:3]:  # Lấy tối đa 3 sự kiện đầu tiên để minh họa
        target_f = ev.get('start_frame', 0)
        cap.set(cv2.CAP_PROP_POS_FRAMES, target_f)
        ret, frame = cap.read()
        if ret:
            rgb_f = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            alert_frames.append((ev, rgb_f))
    cap.release()
    
    if alert_frames:
        plt.figure(figsize=(15, 5 * len(alert_frames)))
        for idx, (ev, img) in enumerate(alert_frames):
            plt.subplot(len(alert_frames), 1, idx + 1)
            plt.imshow(img)
            plt.title(f"Sự kiện #{ev['event_id']} lúc {ev['timestamp_str']} [{ev['level']}] - Điểm tin cậy: {ev['score']:.2f}", fontsize=12, color='red')
            plt.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print("ℹ️ Chưa có sự kiện cảnh báo nào để hiển thị ảnh snapshot.")

In [3]:
# =====================================================================
# 5. KHỞI CHẠY GIAO DIỆN WEB TRỰC QUAN (GRADIO INTERACTIVE UI)
# =====================================================================

from demo.app import build_app

print("🌐 Đang khởi động giao diện Web UI Demo...")
demo_app = build_app()
# Chạy trực tiếp trong notebook hoặc mở cổng localhost:7860
demo_app.launch(inline=True, share=False)

ModuleNotFoundError: No module named 'demo'